# Non-Linear Economy Estimation (ANN)

Please note that this notebook uses a venv which points to a base python version of **3.13**, some functionality may be limited if using an older version of python.

## All Imports

In [1]:
%pip install pandas numpy torch --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
%matplotlib widget

## Data From Source Package

In [3]:
%pip install -e ../ --quiet

Note: you may need to restart the kernel to use updated packages.


In [4]:
import autonomous_fed as afed

In [5]:
le_solver = afed.EnvironmentSolver(fred_key="7ab121fb17773e187bb6508e83e411da")

# Data Check
print (le_solver.historical_data.head())
print (le_solver.historical_data.tail())

             pi         y     i
date                           
1987Q3  2.66122 -0.388640  6.84
1987Q4  2.91876  0.526730  6.92
1988Q1  3.06577  0.260731  6.66
1988Q2  3.35274  0.788853  7.16
1988Q3  3.80207  0.595530  7.98
             pi         y     i
date                           
2006Q2  3.35642  1.319357  4.91
2006Q3  3.13805  0.978513  5.25
2006Q4  2.66316  1.380249  5.25
2007Q1  2.91019  1.210046  5.26
2007Q2  2.72883  1.336951  5.25


## Model Architecture

### Single Hidden Layer NARX Model

For our economy transition equations we have: ${y_t=\hat{f}^y(y_{t-1},y_{t-2},\pi_t,\pi_{t-1},\pi_{t-2},i_t,i_{t-1},i_{t-2})+\epsilon_t^y}$ and ${\pi_t=\hat{f}^\pi(y_t,y_{t-1},y_{t-2},\pi_{t-1},\pi_{t-2},i_t,i_{t-1},i_{t-2})+\epsilon_t^\pi}$

In this scenario our predictor function, ${\hat{f}}$ is an ANN (Artificial Neural Network) but it can be swapped with other nonlinear functions such as a sigmoid function or wavelet network. For our ANN predictor we have ${\hat{f}^m=b_0^m+\sum_{j=1}^h v_j^mG(\omega_j^{m'}s_t^m+b_j^m), m\in\{y,\pi}\}$

The Components of the ANN are as follows:
- ${m}$: ${\{y,\pi}\}$
- ${s_t^m}$: Input state vectors at time t.
- ${w_j^m}$: Weight vector for the j-th hidden neuron.
- ${b_j^m}$: Bias term for the current neuron.
- ${G(\cdot)}$: Activation function (nonlinear transform).
- ${v_j^m}$: Weight from hidden neuron ${j}$ to the output layer
- ${b_0^m}$: Bias at the output layer
- ${h}$: Number of hidden neurons.

As seen above the Neural Network type is a NARX Model with a single hidden layer and the activation function is the hyperbolic tangent therefore we define ${G(\cdot)}$ as follows: ${G(x)=tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}}$

As done in the reference paper the ANNs are intialized with Nguyen-Widrow Initialization. In addition to this we use the Levenberg-Marquardt Algorithm for our optimizer.

### Model Buildout

in the following cells we will replicate the Bundesbank's Neural Net as close as possible by rebuilding some of the MATLAB tools in python for use with PyTorch.

In [6]:
from typing import Iterable, Dict, Union, Tuple, Any
import random
from dataclasses import dataclass
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from torch.utils.data import TensorDataset

In [7]:
DTYPE = torch.float64

torch.use_deterministic_algorithms(True)

# (Optional if you ever use CUDA)
# torch.backends.cuda.matmul.allow_tf32 = False
# torch.backends.cudnn.allow_tf32 = False

#### Pre-Tensor-Construction Scaling

In [8]:
@dataclass
class MapMinMax:
    """
    Min-max scaler that maps data to a specified range.
    
    Attributes:
        in_min (np.ndarray): Minimum values for each feature in the input data.
        in_max (np.ndarray): Maximum values for each feature in the input data.
        out_lo (float): Lower bound of the output range.
        out_hi (float): Upper bound of the output range.
    
    Methods:
        fit: Compute the min and max values for scaling.
        transform: Scale the input data to the specified range.
        inverse_transform: Revert the scaled data back to the original range.
    """
    in_min: np.ndarray = None
    in_max: np.ndarray = None
    out_lo: float = -1.0
    out_hi: float =  1.0
    # precomputed
    scale_: np.ndarray = None
    mid_:   np.ndarray = None

    def fit(self, X: np.ndarray) -> "MapMinMax":
        """
        Fit the scaler to the data.

        Args:
            X (np.ndarray): Input data to compute min and max values.

        Returns:
            MapMinMax: The fitted scaler instance.

        Raises:
            ValueError: If X is empty or not a 2D array.
        """

        self.in_min = np.nanmin(X, axis=0)
        self.in_max = np.nanmax(X, axis=0)
        # handle constant columns robustly
        rng = np.where((self.in_max - self.in_min) == 0.0, 1.0, (self.in_max - self.in_min))
        self.scale_ = (self.out_hi - self.out_lo) / rng
        self.mid_   = (self.out_hi + self.out_lo)/2.0 - self.scale_ * (self.in_max + self.in_min)/2.0
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        """
        Scale the input data to the specified range.

        Args:
            X (np.ndarray): Input data to be scaled.

        Returns:
            np.ndarray: Scaled data.

        Raises:
            ValueError: If the scaler has not been fitted.
        """
        return self.scale_ * X + self.mid_

    def inverse_transform(self, Xs: np.ndarray) -> np.ndarray:
        """
        Revert the scaled data back to the original range.

        Args:
            Xs (np.ndarray): Scaled data to be reverted.

        Returns:
            np.ndarray: Data in the original scale.

        Raises:
            ValueError: If the scaler has not been fitted.
        """
        # invert: X = (Xs - mid_) / scale_
        inv_scale = np.where(self.scale_ == 0.0, 1.0, self.scale_)
        return (Xs - self.mid_) / inv_scale

#### Nguyen-Widrow Initialization

In [9]:
@torch.no_grad()
def nguyen_widrow_(layer: nn.Linear) -> None:
    """
    Nguyen-Widrow initialization for a single linear layer feeding a tanh/sigmoid block. Scales incoming weight vectors to beta = 0.7 * (n_out)^(1/n_in). Bias in [-beta, beta].
    
    Args:
        layer (nn.Linear): Linear layer to initialize.

    Returns:
        None

    Raises:
        None
    """
    n_out, n_in = layer.weight.shape
    with torch.no_grad():
        W = torch.empty_like(layer.weight).uniform_(-0.5, 0.5)
        norms = torch.norm(W, dim=1, keepdim=True).clamp_min(1e-12)
        W = W / norms
        beta = 0.7 * (n_out ** (1.0 / max(1, n_in)))
        layer.weight.copy_(W * beta)
        layer.bias.uniform_(-beta, beta)

#### Levenberg-Marquardt Optimizer

In [10]:
class LevenbergMarquardt(torch.optim.Optimizer):
    """
    Implements the Levenberg-Marquardt optimization algorithm for non-linear least squares problems.

    Attributes:
        params (Iterable[torch.nn.Parameter]): Iterable of parameters to optimize.
        mu (float): Initial damping factor.
        mu_lo (float): Minimum damping factor.
        mu_hi (float): Maximum damping factor.
        mu_up (float): Factor to increase mu when step is rejected.
        mu_down (float): Factor to decrease mu when step is accepted.
        weight_decay (float): Weight decay (L2 penalty) coefficient.

    Methods:
        step(closure): Performs a single optimization step.
    """
    def __init__(self, params: Iterable[torch.nn.Parameter], mu: float=1e-3, mu_lo: float=1e-12, mu_hi: float=1e10, mu_up: float=10.0, mu_down: float=0.1, weight_decay: float=0.0) -> None:
        """
        Initializes the Levenberg-Marquardt optimizer.

        Args:
            params (Iterable[torch.nn.Parameter]): Iterable of parameters to optimize.
            mu (float, optional): Initial damping factor. Default is 1e-3.
            mu_lo (float, optional): Minimum damping factor. Default is 1e-12
            mu_hi (float, optional): Maximum damping factor. Default is 1e10.
            mu_up (float, optional): Factor to increase mu when step is rejected. Default is 10.0.
            mu_down (float, optional): Factor to decrease mu when step is accepted. Default is 0.1.
            weight_decay (float, optional): Weight decay (L2 penalty) coefficient. Default is 0.0.

        Returns:
            None

        Raises:
            ValueError: If any of the hyperparameters are out of valid range.
        """
        defaults = dict(mu=mu, mu_lo=mu_lo, mu_hi=mu_hi, mu_up=mu_up, mu_down=mu_down, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure: callable) -> Dict[str, Union[float, bool]]:
        """
        Performs a single optimization step using the Levenberg-Marquardt algorithm.

        Args:
            closure (callable): A closure that reevaluates the model and returns the residuals.

        Returns:
            Dict: A dictionary containing information about the optimization step:
                - "rho": Gain ratio (float).
                - "accepted": Whether the step was accepted (bool).
                - "mu": Updated damping factor (float).
                - "old_sse": Sum of squared errors before the step (float).
                - "new_sse": Sum of squared errors after the step (float).

        Raises:
            RuntimeError: If the linear system cannot be solved.
        """
        # Flatten parameters
        params = []
        for group in self.param_groups:
            for p in group['params']:
                if p.requires_grad:
                    params.append(p)
        theta = parameters_to_vector(params)
        P = theta.numel()

        # Compute residuals r(theta)
        r = closure() # 1D vector (M,)
        if r.dim() > 1:
            r = r.reshape(-1)
        M = r.numel()

        # Build J explicitly (M x P) by autograd on each residual entry
        # NOTE: This is slow; fine for small M,P only.
        J = torch.zeros(M, P, dtype=theta.dtype, device=theta.device)
        # We need a graph for grads w.r.t. params
        # Recompute r with graph (not no_grad)
        with torch.enable_grad():
            # Recompute r with graph
            rr = closure()
            rr = rr.reshape(-1)
            for i in range(M):
                self.zero_grad(set_to_none=True)
                grad_i = torch.autograd.grad(rr[i], params, retain_graph=True, allow_unused=False)
                J[i] = parameters_to_vector(grad_i)

        # Form normal equations: (J^T J + mu I) Δ = -J^T r
        group = self.param_groups[0]
        mu = group['mu']
        wd = group['weight_decay']

        JT = J.transpose(0, 1) # P x M
        JTJ = JT @ J # P x P
        if wd != 0.0:
            JTJ = JTJ + wd * torch.eye(P, device=theta.device, dtype=theta.dtype)
        rhs = -(JT @ r) # P

        A = JTJ + mu * torch.eye(P, device=theta.device, dtype=theta.dtype)
        # Solve
        try:
            dtheta = torch.linalg.solve(A, rhs)
        except RuntimeError:
            # fallback to least-squares
            dtheta, *_ = torch.linalg.lstsq(A, rhs.unsqueeze(1))
            dtheta = dtheta.squeeze(1)

        # Evaluate gain ratio ρ to adapt mu
        old_r = r
        old_sse = 0.5 * (old_r @ old_r)

        # Canonical LM quantities BEFORE updating parameters
        g = JT @ old_r # gradient (P,)
        predicted_reduction = 0.5 * (dtheta @ (mu * dtheta - g))

        # If model predicts no reduction (numerical or negative), force μ increase
        if predicted_reduction <= 0:
            rho = -1.0
            # Reject immediately: increase μ and keep params
            new_mu = min(group['mu_hi'], max(group['mu_lo'], mu * group['mu_up']))
            group['mu'] = new_mu
            return {
                "rho": float(rho),
                "accepted": False,
                "mu": float(group['mu']),
                "old_sse": float(old_sse.item()),
                "new_sse": float(old_sse.item())
            }

        # Tentative update
        new_theta = theta + dtheta
        old_params = theta.clone()
        vector_to_parameters(new_theta, params)

        new_r = closure().reshape(-1)
        new_sse = 0.5 * (new_r @ new_r)

        rho = (old_sse - new_sse) / predicted_reduction

        if rho > 0 and new_sse < old_sse: # accept
            new_mu = min(group['mu_hi'], max(group['mu_lo'], mu * group['mu_down']))
            group['mu'] = new_mu
            success = True
        else: # reject
            vector_to_parameters(old_params, params)
            new_mu = min(group['mu_hi'], max(group['mu_lo'], mu * group['mu_up']))
            group['mu'] = new_mu
            success = False
            new_sse = old_sse # report unchanged SSE

        return {
            "rho": float(rho.item()),
            "accepted": success,
            "mu": float(group['mu']),
            "old_sse": float(old_sse.item()),
            "new_sse": float(new_sse.item())
        }

#### Single Hidden Layer Artificial Neural Network

In [ ]:
class SingleHiddenLayerNet(nn.Module):
    """
    Single hidden layer neural network.
    
    Attributes:
        h (nn.Linear): Hidden layer.
        o (nn.Linear): Output layer.

    Methods:
        forward: Forward pass through the network.
    """
    def __init__(self, n_in: int, n_hidden: int, n_out: int) -> None:
        super().__init__()
        self.h = nn.Linear(n_in, n_hidden)
        nguyen_widrow_(self.h) # only hidden layer
        self.o = nn.Linear(n_hidden, n_out)
        # Small random init for output layer (paper-style)
        bound = 0.03
        nn.init.uniform_(self.o.weight, -bound, bound)
        nn.init.uniform_(self.o.bias, -bound, bound)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the network.
        
        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output tensor after passing through the network.

        Raises:
            None
        """
        if x.dtype != self.h.weight.dtype:
            x = x.to(self.h.weight.dtype)
        return self.o(torch.tanh(self.h(x)))

#### Lagged Data Matrix

In [12]:
def make_lag_matrix(df: pd.DataFrame, target_col: str, holdout_frac: float=0.15, feature_range: Tuple[float, float]=(-1.0, 1.0)) -> Tuple[TensorDataset, TensorDataset, Dict[str, Any]]:
    """
    Constructs lagged input-output data matrices for time series modeling.

    Args:
        df (pd.DataFrame): DataFrame containing the time series data.
        target_col (str): Name of the target column to predict. Must be either 'y' or 'pi'.
        holdout_frac (float, optional): Fraction of data to hold out for validation. Default is 0.15.
        feature_range (Tuple[float, float], optional): Desired range of transformed features. Default is (-1.0, 1.0).

    Returns:
        Tuple[TensorDataset, TensorDataset, Dict[str, Any]]: Training and validation datasets as TensorDataset objects.

    Raises:
        ValueError: If target_col is not 'y' or 'pi', or if required columns are missing.
    """
    required = {"y", "pi", "i"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"DataFrame missing required columns: {missing}")

    data = df.copy().sort_index()

    # Create needed lags
    data["y_lag1"]  = data["y"].shift(1)
    data["y_lag2"]  = data["y"].shift(2)
    data["pi_lag1"] = data["pi"].shift(1)
    data["pi_lag2"] = data["pi"].shift(2)
    data["i_lag1"]  = data["i"].shift(1)
    data["i_lag2"]  = data["i"].shift(2)

    if target_col == "y":
        feature_cols = [
            "y_lag1",
            "pi_lag1",
            "i_lag1", 
            "i_lag2"
        ]
        y_series = data["y"]
    elif target_col == "pi":
        feature_cols = [
            "y", 
            "y_lag1", 
            "y_lag2",
            "pi_lag1", 
            "pi_lag2",
            "i_lag1"
        ]
        y_series = data["pi"]
    else:
        raise ValueError("target_col must be 'y' or 'pi' for enforced paper design.")

    X_df = data[feature_cols]
    mask = X_df.notna().all(axis=1) & y_series.notna()
    X = X_df[mask].values.astype(np.float64)
    y = y_series[mask].values.astype(np.float64).reshape(-1, 1)

    # deterministic “last 15%” validation split
    n_total = len(X)
    n_hold  = int(np.floor(holdout_frac * n_total))
    n_train = n_total - n_hold
    Xtr_raw, Xva_raw = X[:n_train], X[n_train:]
    ytr_raw, yva_raw = y[:n_train], y[n_train:]

    # fit scalers on TRAIN ONLY
    x_scaler = MapMinMax(out_lo=feature_range[0], out_hi=feature_range[1]).fit(Xtr_raw)
    y_scaler = MapMinMax(out_lo=feature_range[0], out_hi=feature_range[1]).fit(ytr_raw)

    Xtr = x_scaler.transform(Xtr_raw).astype(np.float64)
    Xva = x_scaler.transform(Xva_raw).astype(np.float64)
    ytr = y_scaler.transform(ytr_raw).astype(np.float64)
    yva = y_scaler.transform(yva_raw).astype(np.float64)

    train_ds = TensorDataset(torch.from_numpy(Xtr).to(DTYPE), torch.from_numpy(ytr).to(DTYPE))
    val_ds   = TensorDataset(torch.from_numpy(Xva).to(DTYPE), torch.from_numpy(yva).to(DTYPE))

    meta = {
        "feature_cols": feature_cols,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "Xtr_raw": Xtr_raw, 
        "Xva_raw": Xva_raw,
        "ytr_raw": ytr_raw, 
        "yva_raw": yva_raw
    }
    return train_ds, val_ds, meta

#### Training Function

In [13]:
def train_one_run_LM(model: nn.Module, train_ds: TensorDataset, val_ds: TensorDataset, max_epochs: int=200, patience: int=6, mu_init: float=1e-3) -> Tuple[float, nn.Module]:
    """
    Trains a neural network model using the Levenberg-Marquardt optimization algorithm with early stopping based on validation loss.

    Args:
        model (nn.Module): Neural network model to train.
        train_ds (TensorDataset): Training dataset.
        val_ds (TensorDataset): Validation dataset.
        max_epochs (int, optional): Maximum number of training epochs. Default is 200.
        patience (int, optional): Number of epochs to wait for improvement before early stopping. Default is 25.
        mu_init (float, optional): Initial damping factor for LM optimizer. Default is 1e-3.

    Returns:
        Tuple[float, nn.Module]: Best validation MSE and the trained model.

    Raises:
        None
    """
    Xtr, ytr = train_ds.tensors
    Xva, yva = val_ds.tensors
    model = model.to(DTYPE)
    
    # Levenberg–Marquardt optimizer instance
    opt = LevenbergMarquardt(model.parameters(), mu=mu_init)
    loss_fn = torch.nn.MSELoss()
    
    best = {"val": float("inf"), "state": None, "epoch": -1}
    bad_epochs = 0
    
    # LM closure returning residual vector (NOT scalar loss)
    def closure_train():
        model.zero_grad(set_to_none=True)
        pred = model(Xtr)
        r = (pred - ytr).reshape(-1)
        return r
    
    for ep in range(max_epochs):
        # One LM parameter update
        info = opt.step(closure_train)
        
        # Evaluate on validation set
        model.eval()
        with torch.no_grad():
            val_pred = model(Xva)
            val_mse = loss_fn(val_pred, yva).item()
        
        # Early stopping logic
        if val_mse < best["val"] - 1e-8:
            best.update(state={k: v.clone() for k, v in model.state_dict().items()},
                        val=val_mse, epoch=ep)
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

        # Optional monitoring
        if (ep+1) % 10 == 0:
            print(f"Epoch {ep+1:03d}: val={val_mse:.6f}, μ={info['mu']:.2e}, accepted={info['accepted']}")
    
    # Restore best parameters
    if best["state"] is not None:
        model.load_state_dict(best["state"])
    
    return best["val"], model

#### Hidden Search Function:

In [ ]:
def search_hidden_units_LM(df: pd.DataFrame, target_col: str, hidden_grid: range = range(1, 11), seeds: int=30, holdout_frac: float=0.15) -> Dict[str, Union[float, int, nn.Module]]:
    """
    Searches for the optimal number of hidden units in a single hidden layer neural network using the Levenberg-Marquardt optimization algorithm.

    Args:
        df (pd.DataFrame): DataFrame containing the time series data.
        target_col (str): Name of the target column to predict. Must be either 'y' or 'pi'.
        hidden_grid (range, optional): Range of hidden units to search over. Default is range(1, 11).
        seeds (int, optional): Number of random initializations per hidden unit setting. Default is 30.
        holdout_frac (float, optional): Fraction of data to hold out for validation. Default is 0.15.

    Returns:
        Dict[str, Union[float, int, nn.Module]]: Dictionary containing the best overall
            - "metric": Best validation MSE (float).
            - "h": Number of hidden units for the best model (int).
            - "model": The best trained model (nn.Module).

    Raises:
        ValueError: If target_col is not 'y' or 'pi', or if required columns are missing.
    """
    overall = {"metric": float("inf")}
    per_h_stats = []
    for h in hidden_grid:
        mses, runs = [], []

        for s in range(seeds):
            seed = 10_000 + 97*h + s
            torch.manual_seed(seed)
            np.random.seed(seed)
            random.seed(seed)
            
            train_ds, val_ds, meta = make_lag_matrix(
                df, target_col=target_col, holdout_frac=holdout_frac
            )
            n_in = train_ds.tensors[0].shape[1]
            
            model = SingleHiddenLayerNet(n_in=n_in, n_hidden=h, n_out=1).to(dtype=DTYPE)
            
            val_mse, fitted = train_one_run_LM(model, train_ds, val_ds)
            mses.append(val_mse)
            runs.append((val_mse, fitted, meta))
        mean_mse = float(np.mean(mses))
        per_h_stats.append((h, mean_mse, mses))
        if mean_mse < overall["metric"]:
            # pick the single best run *within this h* to keep
            best_run = min(runs, key=lambda t: t[0])
            overall = {
                "metric": mean_mse, 
                "h": h, 
                "model": best_run[1],
                "meta": best_run[2]
            }
    
    print("\n=== Overall Best ===")
    print(f"h = {overall['h']},  val MSE = {overall['metric']:.6f}")
    return overall

#### Invert Pre-Tensor-Construction Scaling

In [15]:
@torch.no_grad()
def predict_inverse(model: nn.Module, X_raw: np.ndarray, x_scaler: MapMinMax, y_scaler: MapMinMax, device: str="cpu") -> np.ndarray:
    """
    Predict using the model and inverse transform the output to the original scale.

    Args:
        model (nn.Module): Trained PyTorch model.
        X_raw (np.ndarray): Raw input features.
        x_scaler (MapMinMax): Scaler for input features.
        y_scaler (MapMinMax): Scaler for output.
        device (str, optional): Device to run the model on. Default is "cpu".

    Returns:
        np.ndarray: Predicted values in the original scale.

    Raises:
        None
    """
    Xs = x_scaler.transform(X_raw).astype(np.float64)
    X_tensor = torch.from_numpy(Xs).to(dtype=DTYPE, device=device)
    yhat_s = model.to(dtype=DTYPE, device=device)(X_tensor)
    yhat_s_cpu = yhat_s.detach().to("cpu").numpy()
    yhat = y_scaler.inverse_transform(yhat_s_cpu)
    return yhat

### Train, Search, and Invert 

In [16]:
best_y  = search_hidden_units_LM(le_solver.historical_data, target_col='y')
best_pi = search_hidden_units_LM(le_solver.historical_data, target_col='pi')

y_meta  = best_y["meta"]
pi_meta = best_pi["meta"]

device = "cpu"
yhat_val  = predict_inverse(best_y["model"],  y_meta["Xva_raw"],  y_meta["x_scaler"],  y_meta["y_scaler"],  device=device)
pihat_val = predict_inverse(best_pi["model"], pi_meta["Xva_raw"], pi_meta["x_scaler"], pi_meta["y_scaler"], device=device)

Epoch 010: val=0.014635, μ=1.00e-03, accepted=True
Epoch 010: val=0.024803, μ=1.00e-01, accepted=False
Epoch 010: val=0.013234, μ=1.00e-07, accepted=True
Epoch 010: val=0.013181, μ=1.00e-03, accepted=True
Epoch 010: val=0.013228, μ=1.00e-05, accepted=True
Epoch 010: val=0.014046, μ=1.00e-05, accepted=True
Epoch 010: val=0.013234, μ=1.00e-07, accepted=True
Epoch 010: val=0.013234, μ=1.00e-09, accepted=True
Epoch 010: val=0.013235, μ=1.00e-07, accepted=True
Epoch 010: val=0.013179, μ=1.00e-05, accepted=True
Epoch 010: val=0.013234, μ=1.00e-09, accepted=True
Epoch 010: val=0.025783, μ=1.00e-03, accepted=True
Epoch 020: val=0.013234, μ=1.00e-09, accepted=True
Epoch 010: val=0.013234, μ=1.00e-07, accepted=True
Epoch 010: val=0.012751, μ=1.00e-03, accepted=True
Epoch 010: val=0.015517, μ=1.00e-01, accepted=True
Epoch 020: val=0.014104, μ=1.00e-01, accepted=True
Epoch 010: val=0.013233, μ=1.00e-05, accepted=True
Epoch 010: val=0.012844, μ=1.00e-03, accepted=True
Epoch 010: val=0.013232, μ=1.0

### Figure Replication